In [ ]:
from pygom.model.model_spec import ModelSpec
from pygom.model.transition import Transition
from pygom.model.pygom_model import Model

In [ ]:
living_states = ['S', 'E', 'I', 'R']
dead_states = ['D']
states = living_states + dead_states

params = ['beta', 'alpha', 'gamma', 'mu_b', 'mu_d', 'p']
derived_params = [('N', 'S+E+I+R')]

inf = Transition(origin='S', destination='E', rate='beta*S*I/N')
prog = Transition(origin='E', destination='I', rate='alpha*E')
rec = Transition(origin='I', destination='R', rate='gamma*I*(1-p)')
fatal = Transition(origin='I', destination='D', rate='gamma*I*p')

birth = Transition(destination='S', rate='mu_b*N')
deaths = [Transition(origin=state, rate=f'mu_d*{state}') for state in living_states]

event_list = [inf, prog, rec, fatal, birth] + deaths

mod = Model(state=states, param=params, derived_param=derived_params, event=event_list)

# Maths

## Foundations

In [50]:
mod.state_change_matrix.get_equation()

Matrix([
[-1,  0,  0,  0, 1, -1,  0,  0,  0],
[ 1, -1,  0,  0, 0,  0, -1,  0,  0],
[ 0,  1, -1, -1, 0,  0,  0, -1,  0],
[ 0,  0,  1,  0, 0,  0,  0,  0, -1],
[ 0,  0,  0,  1, 0,  0,  0,  0,  0]])

In [51]:
mod.ode.get_equation()

Matrix([
[                 -I*S*beta/N + N*mu_b - S*mu_d],
[                -E*alpha - E*mu_d + I*S*beta/N],
[E*alpha - I*gamma*p - I*gamma*(1 - p) - I*mu_d],
[                      I*gamma*(1 - p) - R*mu_d],
[                                     I*gamma*p]])

In [52]:
mod.event_rate_vector.get_equation()

Matrix([
[     I*S*beta/N],
[        E*alpha],
[I*gamma*(1 - p)],
[      I*gamma*p],
[         N*mu_b],
[         S*mu_d],
[         E*mu_d],
[         I*mu_d],
[         R*mu_d]])

## Jacobians

In [53]:
mod.ode_jacobian_params.get_equation()

Matrix([
[-I*S/N,  0,                0, N, -S,        0],
[ I*S/N, -E,                0, 0, -E,        0],
[     0,  E, -I*p - I*(1 - p), 0, -I,        0],
[     0,  0,        I*(1 - p), 0, -R, -I*gamma],
[     0,  0,              I*p, 0,  0,  I*gamma]])

In [54]:
mod.ode_jacobian_states.get_equation()

Matrix([
[-I*beta/N - mu_d,             0,                       -S*beta/N,     0, 0],
[        I*beta/N, -alpha - mu_d,                        S*beta/N,     0, 0],
[               0,         alpha, -gamma*p - gamma*(1 - p) - mu_d,     0, 0],
[               0,             0,                   gamma*(1 - p), -mu_d, 0],
[               0,             0,                         gamma*p,     0, 0]])

In [55]:
mod.event_rates_jacobian_params.get_equation()

Matrix([
[I*S/N, 0,         0, 0, 0,        0],
[    0, E,         0, 0, 0,        0],
[    0, 0, I*(1 - p), 0, 0, -I*gamma],
[    0, 0,       I*p, 0, 0,  I*gamma],
[    0, 0,         0, N, 0,        0],
[    0, 0,         0, 0, S,        0],
[    0, 0,         0, 0, E,        0],
[    0, 0,         0, 0, I,        0],
[    0, 0,         0, 0, R,        0]])

In [56]:
mod.event_rates_jacobian_states.get_equation()

Matrix([
[I*beta/N,     0,      S*beta/N,    0, 0],
[       0, alpha,             0,    0, 0],
[       0,     0, gamma*(1 - p),    0, 0],
[       0,     0,       gamma*p,    0, 0],
[       0,     0,             0,    0, 0],
[    mu_d,     0,             0,    0, 0],
[       0,  mu_d,             0,    0, 0],
[       0,     0,          mu_d,    0, 0],
[       0,     0,             0, mu_d, 0]])

## Hessians

In [24]:
mod.ode_hessian_params.get_equation()

[[[0, 0, 0], [0, 0, 0], [0, 0, 0]], [[0, 0, 0], [0, 0, 0], [0, 0, 0]], [[0, 0, 0], [0, 0, 0], [0, 0, 0]]]

In [25]:
mod.ode_hessian_states.get_equation()

[[[0, -beta/N, 0], [-beta/N, 0, 0], [0, 0, 0]], [[0, beta/N, 0], [beta/N, 0, 0], [0, 0, 0]], [[0, 0, 0], [0, 0, 0], [0, 0, 0]]]

In [26]:
mod.ode_mixed_hessian_states_params.get_equation()

[[[-I/N, 0, -1], [-S/N, 0, 0], [0, 0, 0]], [[I/N, 0, 0], [S/N, -1, -1], [0, 0, 0]], [[0, 0, 0], [0, 1, 0], [0, 0, -1]]]

In [27]:
mod.event_rates_hessian_params.get_equation()

[[[0, 0, 0], [0, 0, 0], [0, 0, 0]], [[0, 0, 0], [0, 0, 0], [0, 0, 0]], [[0, 0, 0], [0, 0, 0], [0, 0, 0]], [[0, 0, 0], [0, 0, 0], [0, 0, 0]], [[0, 0, 0], [0, 0, 0], [0, 0, 0]], [[0, 0, 0], [0, 0, 0], [0, 0, 0]]]

In [28]:
mod.event_rates_hessian_states.get_equation()

[[[0, beta/N, 0], [beta/N, 0, 0], [0, 0, 0]], [[0, 0, 0], [0, 0, 0], [0, 0, 0]], [[0, 0, 0], [0, 0, 0], [0, 0, 0]], [[0, 0, 0], [0, 0, 0], [0, 0, 0]], [[0, 0, 0], [0, 0, 0], [0, 0, 0]], [[0, 0, 0], [0, 0, 0], [0, 0, 0]]]

In [29]:
mod.event_rates_mixed_hessian_states_params.get_equation()

[[[I/N, 0, 0], [S/N, 0, 0], [0, 0, 0]], [[0, 0, 0], [0, 1, 0], [0, 0, 0]], [[0, 0, 0], [0, 0, 0], [0, 0, 0]], [[0, 0, 1], [0, 0, 0], [0, 0, 0]], [[0, 0, 0], [0, 0, 1], [0, 0, 0]], [[0, 0, 0], [0, 0, 0], [0, 0, 1]]]